read json data

In [0]:
df = spark.read.format("json")\
          .option("inferSchema","true")\
          .option("multiline","true")\
          .load("/Volumes/event_driven_new/stream/streaming/jsonsource")
df.display()

In [0]:
df.printSchema()

In [0]:
df.select("order_id","timestamp","customer").display()

In [0]:
df.select("items","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.postal_code","customer.address.country").display()


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
df = df.withColumn("items",explode_outer("items"))
df.display()

In [0]:
df = df.select("items.item_id","items.product_name","items.price","items.quantity","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.postal_code","customer.address.country","payment.method","payment.transaction_id","metadata")

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
df = df.withColumn("metadata",explode_outer("metadata"))
df.display()

In [0]:
df= df.select("*","metadata.key","metadata.value").drop("metadata")
df.display()

In [0]:
my_schema = """
    order_id STRING,
    timestamp STRING,
    customer STRUCT<
      customer_id: STRING,
      name: STRING,
      email: STRING,
      address: STRUCT<
        city: STRING,
        postal_code: STRING,
        country: STRING
      >
    >,
    payment STRUCT<
      method: STRING,
      transaction_id: STRING
    >,
    items ARRAY<STRUCT<
      item_id: STRING,
      product_name: STRING,
      price: DOUBLE,
      quantity: LONG
    >>,
    metadata ARRAY<STRUCT<
      key: STRING,
      value: STRING
    >>
"""

In [0]:
#spark.conf.set("spark.sql.streaming.schemaInference", "true")

df1 = spark.readStream.format("json")\
          .option("multiline","true")\
          .schema(my_schema)\
          .load("/Volumes/event_driven_new/stream/streaming/jsonsource")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
df1 = df1.withColumn("items",explode_outer("items"))

In [0]:
df1 = df1.select("items.item_id","items.product_name","items.price","items.quantity","order_id","timestamp","customer.customer_id","customer.name","customer.email","customer.address.city","customer.address.postal_code","customer.address.country","payment.method","payment.transaction_id","metadata")
df1 = df1.withColumn("metadata",explode_outer("metadata"))


df1= df1.select("*","metadata.key","metadata.value").drop("metadata")


In [0]:
df1.writeStream.format("delta")\
    .outputMode("append")\
    .trigger(once=True)\
    .option("path","/Volumes/event_driven_new/stream/streaming/jsonsink")\
    .option("checkpointLocation","/Volumes/event_driven_new/stream/streaming/jsoncheckpoint")\
    .start()


In [0]:
%sql
select * from delta.`/Volumes/event_driven_new/stream/streaming/jsonsink`

In [0]:
df.limit(2).display()

In [0]:
schema_ddl = """
  order_id STRING,
  timestamp TIMESTAMP,
  customer STRUCT<
    customer_id: STRING,
    name: STRING,
    email: STRING,
    address: STRUCT<
      city: STRING,
      postal_code: STRING,
      country: STRING
    >
  >,
  items ARRAY<STRUCT<
    item_id: STRING,
    product_name: STRING,
    quantity: LONG,
    price: DOUBLE
  >>,
  metadata ARRAY<STRUCT<
    key: STRING,
    value: STRING
  >>,
  payment STRUCT<
    method: STRING,
    transaction_id: STRING
  >
"""


In [0]:
df = spark.read.format("json")\
        .option("multiline","true")\
    .schema(schema_ddl)\
    .load("/Volumes/event_driven_new/stream/streaming/jsonsource")
display(df)

In [0]:
df = df.withColumn("items",explode_outer("items"))

In [0]:
df = df.withColumn("metadata",explode_outer("metadata"))

In [0]:
df.display()

In [0]:
df.select("customer.customer_id").display()

In [0]:
df.display()

In [0]:
df.withColumn("oid",when(df.order_id == "ORD1001", "1001")
                    .when(df.order_id == "ORD1001", "1002")
                    .otherwise("none")).display()



In [0]:
df.display()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import collect_list, col

df.groupBy().agg(collect_list(col("order_id"))).display()


In [0]:
df.display()

In [0]:
from pyspark.sql.window import Window
df= df.withColumn("rn",row_number().over(Window.partitionBy("oid").orderBy(col("oid").desc())))

In [0]:
df.display()